# 01 — Filtrado de archivos crudos ENOE

Lee los archivos de las dos carpetas de datos crudos que deben descargarse de Drive  y extrae las columnas
necesarias para el análisis. Los resultados se guardan en `ENOE_filtrado/`.

- `ENOE_crudo/` — trimestres 2020-T2 hasta ultimo periodo disponible.
- `2005-2020T1/` — carpeta compartida con los años anteriores.

Si un archivo ya fue filtrado anteriormente, se omite automáticamente.

In [2]:
import pandas as pd
import os
import re
import gc
import glob

## Rutas

Solo se revisan estas dos carpetas — sin búsqueda recursiva en el resto del Drive.

In [4]:
CARPETAS_ENTRADA = [
    '../data/raw/ENOE_crudo',
    '../data/raw/2005-2020T1', #Carpeta compartida por integrante para hacer un solo filtrado.
]
CARPETA_SALIDA = 'data/ENOE_filtrado'

os.makedirs(CARPETA_SALIDA, exist_ok=True)

for c in CARPETAS_ENTRADA:
    estado = 'OK' if os.path.exists(c) else 'NO ENCONTRADA'
    print(f'  {estado}: {c}')

  OK: ../data/raw/ENOE_crudo
  OK: ../data/raw/2005-2020T1


## Catálogos

Los archivos crudos guardan los campos como números (1, 2, 3...).
Estos diccionarios los convierten a texto legible en el CSV filtrado.

In [5]:
ENTIDADES = {
    1:'Aguascalientes', 2:'Baja California', 3:'Baja California Sur',
    4:'Campeche', 5:'Coahuila', 6:'Colima', 7:'Chiapas', 8:'Chihuahua',
    9:'Ciudad de México', 10:'Durango', 11:'Guanajuato', 12:'Guerrero',
    13:'Hidalgo', 14:'Jalisco', 15:'México', 16:'Michoacán', 17:'Morelos',
    18:'Nayarit', 19:'Nuevo León', 20:'Oaxaca', 21:'Puebla', 22:'Querétaro',
    23:'Quintana Roo', 24:'San Luis Potosí', 25:'Sinaloa', 26:'Sonora',
    27:'Tabasco', 28:'Tamaulipas', 29:'Tlaxcala', 30:'Veracruz',
    31:'Yucatán', 32:'Zacatecas',
}
SEXO      = {1:'Hombre', 2:'Mujer'}
FORMAL    = {1:'Formal', 2:'Informal'}
NIV_INS   = {
    1:'Sin instrucción', 2:'Básica',
    3:'Media superior',  4:'Superior', 5:'No especificado',
}
SECTORES  = {
    1:'Agropecuario',          2:'Industria extractiva',     3:'Manufactura',
    4:'Construcción',          5:'Comercio',                 6:'Restaurantes y hospedaje',
    7:'Transportes y comunicaciones', 8:'Servicios financieros y profesionales',
    9:'Servicios sociales',    10:'Servicios diversos',      11:'Gobierno',
}
OCUPACION = {
    1:'Funcionarios y directivos',   2:'Profesionistas y técnicos',
    3:'Trabajadores de la educación', 4:'Trabajadores del arte',
    5:'Actividades administrativas', 6:'Comerciantes y vendedores',
    7:'Servicios personales',        8:'Actividades agrícolas',
    9:'Industria extractiva y construcción', 10:'Operadores de maquinaria',
    11:'Otras ocupaciones',
}

## Funciones auxiliares

- `parsear_nombre`: extrae año y trimestre del nombre del archivo
- `filtrar`: lee un crudo, se queda solo con ocupados y devuelve las columnas de análisis

In [6]:
def parsear_nombre(ruta):
    # El patrón SDEMT{trim}{YY} aparece en los tres formatos de nombre:
    # SDEMT419.csv, ENOE_SDEMT120.csv, ENOEN_SDEMT422.csv
    nombre = os.path.splitext(os.path.basename(ruta))[0].upper()
    m = re.search(r'SDEMT(\d)(\d{2})$', nombre)
    if not m:
        return None
    trim, yy = int(m.group(1)), m.group(2)
    anio = int('20' + yy)
    if trim not in [1, 2, 3, 4] or not (2000 <= anio <= 2030):
        return None
    return trim, anio


def nombre_filtrado(ruta):
    info = parsear_nombre(ruta)
    if info is None:
        return None
    trim, anio = info
    return f'ENOE_T{trim}{str(anio)[-2:]}.csv'


def a_num(serie):
    # Los archivos de 2023+ guardan algunos campos numéricos como texto
    return pd.to_numeric(serie, errors='coerce')


def filtrar(ruta):
    encabezado = pd.read_csv(ruta, encoding='latin1', nrows=0)
    cols = list(encabezado.columns)

    # El nombre de la columna de estado y del ponderador cambió en 2023
    col_est = 'cve_ent' if 'cve_ent' in cols else 'ent'
    col_fac = 'fac_tri' if 'fac_tri' in cols else 'fac'

    # Solo leemos las columnas que vamos a usar
    quiero = [
        col_est, col_fac, 'sex', 'eda',
        'c_ocu11c', 'rama_est2', 'niv_ins', 'anios_esc',
        'hrsocup', 'ingocup', 'ing_x_hrs', 'emp_ppal', 'clase2',
    ]
    cols_leer = [c for c in quiero if c in cols]
    df = pd.read_csv(ruta, encoding='latin1', low_memory=False, usecols=cols_leer)

    # Quedarse solo con población ocupada
    if 'clase2' in df.columns:
        df = df[a_num(df['clase2']) == 1].copy()
        df.drop(columns=['clase2'], inplace=True)

    # Normalizar tipos numéricos
    for c in [col_est, 'sex', 'c_ocu11c', 'rama_est2', 'niv_ins', 'emp_ppal', 'hrsocup', 'ingocup']:
        if c in df.columns:
            df[c] = a_num(df[c])

    # Unificar nombres para que todos los años queden igual
    df.rename(columns={col_est: 'cve_ent', col_fac: 'fac_tri'}, inplace=True, errors='ignore')

    # Ingreso por hora: usar el campo ya calculado por el INEGI cuando existe,
    # y calcularlo desde ingocup/hrsocup cuando no
    if 'ing_x_hrs' in df.columns:
        df['ingreso_hora'] = a_num(df['ing_x_hrs']).where(a_num(df['ing_x_hrs']) > 0)
    if 'ingocup' in df.columns and 'hrsocup' in df.columns:
        ing, hrs = a_num(df['ingocup']), a_num(df['hrsocup'])
        calculado = (ing / (hrs * 4.33)).round(2)
        if 'ingreso_hora' not in df.columns:
            df['ingreso_hora'] = calculado.where((ing > 0) & (hrs > 0))
        else:
            df['ingreso_hora'] = df['ingreso_hora'].fillna(calculado.where((ing > 0) & (hrs > 0)))

    # Salario semanal a partir del ingreso mensual
    if 'ingocup' in df.columns:
        ing = a_num(df['ingocup'])
        df['salario_semanal'] = (ing / 4.33).round(2).where(ing > 0)

    # Agregar columnas con etiquetas de texto junto a las claves numéricas
    df['estado']     = df['cve_ent'].map(ENTIDADES)  if 'cve_ent'   in df.columns else None
    df['sexo']       = df['sex'].map(SEXO)            if 'sex'       in df.columns else None
    df['sector']     = df['rama_est2'].map(SECTORES)  if 'rama_est2' in df.columns else None
    df['nivel_educ'] = df['niv_ins'].map(NIV_INS)     if 'niv_ins'   in df.columns else None
    df['ocupacion']  = df['c_ocu11c'].map(OCUPACION)  if 'c_ocu11c'  in df.columns else None
    df['formalidad'] = df['emp_ppal'].map(FORMAL)      if 'emp_ppal'  in df.columns else None

    return df

## Proceso principal

Para cada archivo crudo: si el filtrado ya existe se salta, si no se genera y guarda.

In [10]:
# Recopilar archivos de las dos carpetas
archivos = []
for carpeta in CARPETAS_ENTRADA:
    if not os.path.exists(carpeta):
        print(f'  Carpeta no encontrada, se omite: {carpeta}')
        continue
    encontrados = sorted(glob.glob(os.path.join(carpeta, '*SDEMT*.csv')))
    print(f'  {len(encontrados):>3} archivos en {carpeta}')
    archivos.extend(encontrados)

print(f'\nTotal: {len(archivos)} archivos')

   19 archivos en ../data/raw/ENOE_crudo
   21 archivos en ../data/raw/2005-2020T1

Total: 40 archivos


In [11]:
for ruta in archivos:
    nf = nombre_filtrado(ruta)

    if nf is None:
        print(f'  OMITIDO (nombre no reconocido): {os.path.basename(ruta)}')
        continue

    salida = os.path.join(CARPETA_SALIDA, nf)

    if os.path.exists(salida):
        print(f'  ya existe: {nf}')
        continue

    try:
        df = filtrar(ruta)
        df.to_csv(salida, index=False, encoding='utf-8-sig')
        print(f'  {os.path.basename(ruta)} -> {nf}  ({len(df):,} filas, {df.shape[1]} cols)')
        del df
        gc.collect()
    except Exception as e:
        print(f'  ERROR en {os.path.basename(ruta)}: {e}')

print('\nProceso terminado.')

  ya existe: ENOE_T121.csv
  ya existe: ENOE_T122.csv
  ya existe: ENOE_T221.csv
  ya existe: ENOE_T222.csv
  ya existe: ENOE_T320.csv
  ya existe: ENOE_T321.csv
  ya existe: ENOE_T322.csv
  ya existe: ENOE_T420.csv
  ya existe: ENOE_T422.csv
  ya existe: ENOE_T123.csv
  ya existe: ENOE_T124.csv
  ya existe: ENOE_T125.csv
  ya existe: ENOE_T224.csv
  ya existe: ENOE_T225.csv
  ya existe: ENOE_T323.csv
  ya existe: ENOE_T324.csv
  ya existe: ENOE_T325.csv
  ya existe: ENOE_T424.csv
  ya existe: ENOE_T425.csv
  ENOE_SDEMT120.csv -> ENOE_T120.csv  (184,064 filas, 20 cols)
  SDEMT107.csv -> ENOE_T107.csv  (173,235 filas, 20 cols)
  SDEMT116.csv -> ENOE_T116.csv  (167,881 filas, 20 cols)
  SDEMT117.csv -> ENOE_T117.csv  (167,033 filas, 20 cols)
  SDEMT118.csv -> ENOE_T118.csv  (168,166 filas, 20 cols)
  SDEMT206.csv -> ENOE_T206.csv  (172,965 filas, 20 cols)
  SDEMT208.csv -> ENOE_T208.csv  (174,414 filas, 20 cols)
  SDEMT209.csv -> ENOE_T209.csv  (165,603 filas, 20 cols)
  SDEMT211.csv -> 

In [12]:
# Resumen de lo que quedó en la carpeta de salida
filtrados = sorted(os.listdir(CARPETA_SALIDA))
total_mb = 0
print(f'Archivos en ENOE_filtrado/ ({len(filtrados)} total):\n')
for f in filtrados:
    mb = os.path.getsize(os.path.join(CARPETA_SALIDA, f)) / 1_048_576
    total_mb += mb
    print(f'  {f:<26}  {mb:5.1f} MB')
print(f'\n  Total: {total_mb:.1f} MB')

Archivos en ENOE_filtrado/ (40 total):

  ENOE_T107.csv                21.3 MB
  ENOE_T116.csv                20.4 MB
  ENOE_T117.csv                20.3 MB
  ENOE_T118.csv                20.4 MB
  ENOE_T120.csv                22.4 MB
  ENOE_T121.csv                18.1 MB
  ENOE_T122.csv                21.5 MB
  ENOE_T123.csv                24.8 MB
  ENOE_T124.csv                23.5 MB
  ENOE_T125.csv                22.8 MB
  ENOE_T206.csv                21.3 MB
  ENOE_T208.csv                21.4 MB
  ENOE_T209.csv                20.2 MB
  ENOE_T211.csv                20.2 MB
  ENOE_T217.csv                20.7 MB
  ENOE_T218.csv                20.7 MB
  ENOE_T219.csv                21.8 MB
  ENOE_T221.csv                20.3 MB
  ENOE_T222.csv                22.1 MB
  ENOE_T224.csv                23.8 MB
  ENOE_T225.csv                23.0 MB
  ENOE_T306.csv                21.3 MB
  ENOE_T315.csv                20.6 MB
  ENOE_T316.csv                20.4 MB
  ENOE_T317.csv         